# Advanced: Lagrange Multipliers and KKT Conditions for Linear Programs
How do we know that a candidate solution to a linear program is optimal, and where do the algorithms that find one come from? In this advanced notebook, we look under the hood of the [linear programming lecture](CHEME-5800-L5c-Lecture-LinearProgramming-Fall-2026.ipynb). We apply the classical Lagrange multiplier method to a linear program, see why it fails, and repair it with the Karush-Kuhn-Tucker (KKT) conditions.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
>
> * __Explain why equality-only Lagrange multipliers fail for linear programs:__ Form the Lagrangian for a linear program with slack variables and derive its stationarity conditions. Show that leaving out the nonnegativity bounds forces a contradiction unless the objective coefficients all vanish.
> * __State the KKT conditions for a linear program:__ Write the stationarity, primal feasibility, dual feasibility, and complementary slackness conditions using multipliers for the equality and bound constraints. Identify these conditions as necessary and sufficient for optimality in a linear program.
> * __Distinguish the roles of the multipliers:__ Identify which multiplier enforces each constraint and which multipliers must be nonnegative. Explain how complementary slackness pairs each bound with its multiplier so that only active constraints influence the solution.

In this notebook, we first apply the Lagrange multiplier method and identify what goes wrong. We then construct the KKT conditions and close with pointers to the two algorithm families that build on them.

Let's get started!

___


## Lagrange Multipliers
Normally, when faced with a constrained optimization problem, our first thought would be to use the Lagrange multiplier method. So how does the Lagrange multiplier method work for our linear programming problem? Let's find out!

We formulate the Lagrangian function by incorporating the constraints into the objective function using Lagrange multipliers. We then compute the gradient of the Lagrangian function and set it to zero to find the optimal solution. Let's apply this method to our linear programming problem. Suppose we have a linear program in the form:
$$
\begin{align*}
\text{minimize} &\, \sum_{i=1}^{n} c_{i}\;{x}_{i}\\
\text{subject to}~ \mathbf{A}\;\mathbf{x} + \mathbf{s} &= \mathbf{b}\quad\text{where}\,\mathbf{A}\in\mathbb{R}^{m\times n}\,\text{and}\,\mathbf{b}\in\mathbb{R}^{m}\\
x_{i} &\geq 0\quad{i=1,2,\dots,n}\\
s_{j} &\geq 0\quad{j=1,2,\dots,m}
\end{align*}$$
where $\mathbf{s} = (s_1, s_2, \ldots, s_m) \in \mathbb{R}^{m}$ are the _slack variables_ that convert the inequalities into equalities.
* __Why slack variables?__ Working with inequality constraints can be tricky, especially when applying the Lagrange multiplier method, which is typically designed for equality constraints. However, we can convert the inequality constraints into equality constraints by introducing _slack variables_ $s_{j}$ for each of the $m$ constraints.
* __Definition of slack variables__: For the original inequality constraint $\mathbf{a}^{\top}_{i}\cdot \mathbf{x} \leq b_{i}$, the slack variable $s_{i} \geq 0$ represents the amount of "slack" or unused capacity in the constraint. The transformation gives us the equality constraint $\mathbf{a}^{\top}_{i}\cdot \mathbf{x} + s_{i} = b_{i}$ for each constraint $i$. This transformation is crucial because the Lagrange multiplier method requires equality constraints to define the Lagrangian function.

Now, we can write the Lagrangian function $\mathcal{L}(\mathbf{x}, \mathbf{s}, \boldsymbol{\lambda})$ as:
$$
\begin{align*}
\mathcal{L}(\mathbf{x}, \mathbf{s}, \boldsymbol{\lambda}) &= \mathbf{c}^{\top}\mathbf{x} - \boldsymbol{\lambda}^{\top}(\mathbf{A}\cdot\mathbf{x} + \mathbf{s} - \mathbf{b})\\
\end{align*}
$$
where $\boldsymbol{\lambda} \in \mathbb{R}^{m}$ are the Lagrange multipliers associated with the equality constraints. To compute the first-order optimality conditions, we take the gradient of the Lagrangian with respect to $\mathbf{x}$, $\mathbf{s}$, and $\boldsymbol{\lambda}$ and set it to zero:
$$
\begin{align*}
\nabla_{\mathbf{s}}\mathcal{L}(\mathbf{x}, \mathbf{s}, \boldsymbol{\lambda}) &= -\boldsymbol{\lambda} = 0\quad\implies\boldsymbol{\lambda} = 0\quad\text{This is a problem!}\\
\nabla_{\mathbf{x}}\mathcal{L}(\mathbf{x}, \mathbf{s}, \boldsymbol{\lambda}) &= \mathbf{c} - \mathbf{A}^{\top}\boldsymbol{\lambda} = 0\implies\mathbf{c} = 0\quad\text{This is an even bigger problem!}\\
\nabla_{\boldsymbol{\lambda}}\mathcal{L}(\mathbf{x}, \mathbf{s}, \boldsymbol{\lambda}) &= -\left(\mathbf{A}\cdot\mathbf{x} + \mathbf{s} - \mathbf{b}\right) = 0
\end{align*}
$$
From the first equation, the Lagrange multipliers are zero for all constraints, which then (from the second equation) requires $\mathbf{c} = 0$, which is not generally true!

What went wrong? The Lagrangian above includes only the equality constraint. The nonnegativity conditions $\mathbf{x}\geq 0$ and $\mathbf{s}\geq 0$ are inequalities, and the classical Lagrange multiplier method has no way to include them. At the optimum, some of these bounds are active, and leaving them out is what forces the contradiction.

__Hmmmm__. The equality-only Lagrange multiplier method doesn't work for linear programs whose optimum sits on an inequality bound, which is the usual case. We need the _Karush-Kuhn-Tucker (KKT) conditions_ to handle the inequality constraints properly!

___

## Karush-Kuhn-Tucker (KKT) Conditions
The Karush-Kuhn-Tucker (KKT) conditions extend the Lagrange multiplier framework to problems with inequality and non-negativity constraints. For linear programs, these conditions are necessary and sufficient for optimality; for general convex problems, necessity also requires a constraint qualification.

To construct the KKT conditions for a linear program, we need to understand four key concepts:

* __Stationarity__: At the optimum, the competing influences of improving the objective and enforcing the constraints balance out for each decision variable, so that no infinitesimal change in any variable can increase the Lagrangian.
* __Primal feasibility__: The candidate solution must satisfy every original model requirement (every equality condition must hold exactly, and every inequality or non-negativity restriction must be respected).
* __Dual feasibility__: All multipliers that penalize inequality constraints must be non-negative, ensuring that they only oppose constraint violations rather than "reward" them.
* __Complementary slackness__: Every inequality constraint is either exactly tight (active), in which case its multiplier may be positive, or else it is slack (not binding), in which case its multiplier is forced to zero, so that only active constraints influence the solution.

Let's start by rewriting the linear program in standard form, where we introduce slack variables $s\ge0$ for $A\,x\le b$, and flip the maximization problem to a minimization problem by negating the objective coefficients (below, $c$ denotes the negated coefficients):
$$
\begin{aligned}
&\text{Primal LP:}\quad\min_{x,s}\;c^\top x\quad\text{s.t.}\quad
A\,x + s = b,\;x \ge 0,\;s \ge 0.\\
&\text{Lagrange multipliers:}\quad
\lambda\in\mathbb R^m,\;\mu\in\mathbb R^n,\;\nu\in\mathbb R^m,
\quad\mu \ge 0,\;\nu \ge 0.\\
&\boxed{\displaystyle
\mathcal{L}(x,s,\lambda,\mu,\nu)
= c^\top x 
\;-\;\lambda^\top\bigl(A\,x + s - b\bigr)
\;-\;\mu^\top x
\;-\;\nu^\top s}
\end{aligned}
$$
where:
* $\lambda_j$ (unconstrained) enforces the equality constraint $\sum_i A_{ji} x_i + s_j = b_j$,
* $\mu_i \ge 0$ enforces the non-negativity constraint $x_i\ge0$,
* $\nu_j \ge 0$ enforces the non-negativity constraint $s_j\ge0$.

Now, we compute the gradient of the Lagrangian function with respect to the _primal variables_ $(x,s)$ and _dual variable_ $\lambda$, and enforce the complementary slackness conditions for the _dual variables_ $(\mu,\nu)$:
$$
\begin{aligned}
&\nabla_x\mathcal{L}(x,s,\lambda,\mu,\nu) = c - A^\top\lambda - \mu = 0,\\
&\nabla_s\mathcal{L}(x,s,\lambda,\mu,\nu) = -\lambda - \nu = 0,\\
&\nabla_\lambda\mathcal{L}(x,s,\lambda,\mu,\nu) = -\left(A\,x + s - b\right) = 0,\\
&\mu_i x_i = 0,\quad\forall i=1,2,\dots,n,\\
&\nu_j s_j = 0,\quad\forall j=1,2,\dots,m,\\
&\mu_i \ge 0,\quad\forall i=1,2,\dots,n,\\
&\nu_j \ge 0,\quad\forall j=1,2,\dots,m.
\end{aligned}
$$

Great! We have the KKT conditions for the primal linear program. These conditions are necessary and sufficient for optimality in linear programs, and in convex problems that satisfy a constraint qualification. However, what is the actionable algorithm that we can develop from these conditions?

Let's consider two classes of algorithms based on the KKT conditions: [the Revised Simplex algorithm](CHEME-5800-L5c-RevisedSimplex-Algorithm-Fall-2026.ipynb) and the [Interior Point Algorithm](CHEME-5800-L5c-InteriorPointMethod-Algorithm-Fall-2026.ipynb).

___

## Summary
In this notebook, we applied the Lagrange multiplier method to a linear program, saw why the equality-only version fails, and constructed the KKT conditions that replace it.

> __Key Takeaways:__
>
> * __The equality-only Lagrangian breaks down:__ We converted the inequalities to equalities with slack variables and set the gradient of the Lagrangian to zero. The stationarity conditions forced the multipliers and then the objective coefficients to vanish, because the nonnegativity bounds had been left out.
> * __KKT conditions repair the argument:__ We added nonnegative multipliers for the bounds on the decision and slack variables and obtained stationarity, primal feasibility, dual feasibility, and complementary slackness conditions. For a linear program, these conditions are necessary and sufficient for optimality.
> * __Optimality conditions drive the algorithms:__ Complementary slackness pairs each bound with its multiplier, so only active constraints shape the solution. The revised simplex and interior point notebooks develop two algorithm families that search for a point satisfying these conditions.

Certifying optimality means exhibiting multipliers that satisfy these conditions alongside the solution. A solver's optimal status reports that it found such a point within its numerical tolerances.

___